In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'flask-cors', '-q'])
print("✅ flask-cors installed!")

✅ flask-cors installed!


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  app.py  —  Blood Pressure Prediction · Flask Backend
#  Run:  python app.py
#  Open: http://127.0.0.1:5000
# ─────────────────────────────────────────────────────────────────────────────

from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
import joblib
import numpy as np
import os
import json

app = Flask(__name__, static_folder="static")
CORS(app)

# ── Load models ───────────────────────────────────────────────────────────────
MODELS_DIR = "models"

dt_model  = joblib.load(f"{MODELS_DIR}/decision_tree.pkl")
lr_model  = joblib.load(f"{MODELS_DIR}/linear_regression.pkl")
nb_model  = joblib.load(f"{MODELS_DIR}/naive_bayes.pkl")
gb_model  = joblib.load(f"{MODELS_DIR}/gradient_boosting.pkl")
rf_model  = joblib.load(f"{MODELS_DIR}/random_forest.pkl")
scaler    = joblib.load(f"{MODELS_DIR}/scaler.pkl")
le_enc    = joblib.load(f"{MODELS_DIR}/label_encoders.pkl")
le_target = joblib.load(f"{MODELS_DIR}/le_target.pkl")

with open(f"{MODELS_DIR}/accuracy_info.json") as f:
    accuracy_info = json.load(f)

MODEL_MAP = {
    "dt": dt_model,
    "nb": nb_model,
    "gb": gb_model,
    "rf": rf_model,
}


# ── Helper: encode raw form values into feature vector ────────────────────────
def encode_input(data):
    """
    Encoding reference:
        Gender           : M=1, F=0
        Smoking          : No=0, Yes=1
        Alcohol          : High=0, Low=1, Moderate=2
        Physical_Activity: High=0, Low=1, Medium=2
        Stress_Level     : High=0, Low=1, Medium=2
    """
    age         = float(data.get("age", 45))
    gender      = le_enc["Gender"].transform([data.get("gender", "M")])[0]
    bmi         = float(data.get("bmi", 24.5))
    smoking     = le_enc["Smoking"].transform([data.get("smoking", "No")])[0]
    alcohol     = le_enc["Alcohol"].transform([data.get("alcohol", "Low")])[0]
    activity    = le_enc["Physical_Activity"].transform([data.get("activity", "Medium")])[0]
    cholesterol = float(data.get("cholesterol", 190))
    heart_rate  = float(data.get("heart_rate", 72))
    stress      = le_enc["Stress_Level"].transform([data.get("stress", "Low")])[0]

    return np.array([[age, gender, bmi, smoking, alcohol, activity,
                      cholesterol, heart_rate, stress]])


# ── Routes ────────────────────────────────────────────────────────────────────
@app.route("/")
def index():
    """Serve the frontend."""
    return send_from_directory(".", "frontend.html")


@app.route("/api/models", methods=["GET"])
def get_models():
    """Return model accuracy metadata."""
    return jsonify(accuracy_info)


@app.route("/api/predict", methods=["POST"])
def predict():
    """
    POST JSON body:
    {
        "algo": "gb",            // dt | lr | nb | gb | rf
        "age": 45,
        "gender": "M",           // M | F
        "bmi": 24.5,
        "smoking": "No",         // No | Yes
        "alcohol": "Low",        // Low | Moderate | High
        "activity": "Medium",    // Low | Medium | High
        "cholesterol": 190,
        "heart_rate": 72,
        "stress": "Low"          // Low | Medium | High
    }
    """
    try:
        data     = request.get_json()
        algo     = data.get("algo", "gb")
        X_raw    = encode_input(data)
        X_scaled = scaler.transform(X_raw)

        if algo == "lr":
            # Regression → predict Systolic BP → derive category
            sys_bp = float(lr_model.predict(X_scaled)[0])
            dia_bp = sys_bp * 0.65  # approximate diastolic

            if sys_bp < 120 and dia_bp < 80:    cat = "Normal"
            elif sys_bp < 130 and dia_bp < 80:  cat = "Elevated"
            elif sys_bp < 140 or dia_bp < 90:   cat = "High Stage 1"
            else:                                cat = "High Stage 2"

            return jsonify({
                "category":  cat,
                "systolic":  round(sys_bp, 1),
                "diastolic": round(dia_bp, 1),
                "algorithm": accuracy_info["lr"]["name"],
                "accuracy":  accuracy_info["lr"]["accuracy"],
                "metric":    accuracy_info["lr"]["metric"],
            })

        else:
            model    = MODEL_MAP[algo]
            pred_enc = model.predict(X_raw)[0]
            proba    = model.predict_proba(X_raw)[0]
            cat      = le_target.inverse_transform([pred_enc])[0]
            confidence = round(float(proba.max()) * 100, 2)

            bp_ranges = {
                "Normal":       (100, 119, 60, 79),
                "Elevated":     (120, 129, 70, 79),
                "High Stage 1": (130, 139, 80, 89),
                "High Stage 2": (140, 179, 90, 119),
            }
            s_lo, s_hi, d_lo, d_hi = bp_ranges.get(cat, (120, 130, 70, 80))

            return jsonify({
                "category":   cat,
                "systolic":   f"{s_lo}–{s_hi}",
                "diastolic":  f"{d_lo}–{d_hi}",
                "confidence": confidence,
                "algorithm":  accuracy_info[algo]["name"],
                "accuracy":   accuracy_info[algo]["accuracy"],
                "metric":     accuracy_info[algo]["metric"],
                "class_probs": {
                    cls: round(float(p) * 100, 2)
                    for cls, p in zip(le_target.classes_, proba)
                },
            })

    except Exception as e:
        return jsonify({"error": str(e)}), 400


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n  🩺  BP Prediction Server running")
    print("  👉  Open  http://127.0.0.1:5000  in your browser\n")
    app.run(debug=False, port=5000)



  🩺  BP Prediction Server running
  👉  Open  http://127.0.0.1:5000  in your browser

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Apr/2026 12:20:53] "GET / HTTP/1.1" 304 -
127.0.0.1 - - [05/Apr/2026 12:20:53] "GET /api/models HTTP/1.1" 200 -
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
127.0.0.1 - - [05/Apr/2026 12:55:44] "POST /api/predict HTTP/1.1" 200 -
C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardSca

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'app.py'])